In [1]:
# ------------------------------
# Importing necessary libraries
# ------------------------------

import pandas as pd
import tqdm

print("Successfully loaded all libraries!")

Successfully loaded all libraries!


In [ ]:
# ------------------------------
# Loading cleaned NBA data
# ------------------------------

df = pd.read_csv('../data/processed/nba_cleaned_a.csv', dtype={'season': str}, parse_dates=['GAME_DATE_HOME'])

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns")
print(f"Date range: {df['GAME_DATE_HOME'].min().date()} - {df['GAME_DATE_HOME'].max().date()}")
print(f"\nSeasons: {sorted(df['season'].unique())}")

Loaded 15108 rows and 59 columns
Date range: 2014-10-04 - 2025-06-22

Seasons: ['201415', '201516', '201617', '201718', '201819', '201920', '202021', '202122', '202223', '202324', '202425']


In [2]:
# ------------------------------
# Loading cleaned NBA data
# ------------------------------

df = pd.read_csv('../data/processed/nba_cleaned_b.csv', dtype={'season': str}, parse_dates=['GAME_DATE_HOME'])

print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns")
print(f"Date range: {df['GAME_DATE_HOME'].min().date()} - {df['GAME_DATE_HOME'].max().date()}")
print(f"\nSeasons: {sorted(df['season'].unique())}")

Loaded 8048 rows and 59 columns
Date range: 2019-09-30 - 2025-06-22

Seasons: ['201920', '202021', '202122', '202223', '202324', '202425']


In [3]:
# -----------------
# Helper functions
# -----------------

def get_recent_team_stats(df, team, match_date, venue):
    if venue == 'home':
        team_matches = df[(df['TEAM_NAME_HOME'] == team) & (df['GAME_DATE_HOME'] < match_date)]
    elif venue == 'away':
        team_matches = df[(df['TEAM_NAME_AWAY'] == team) & (df['GAME_DATE_HOME'] < match_date)]
    else:
        team_matches = df[((df['TEAM_NAME_HOME'] == team) | (df['TEAM_NAME_AWAY'] == team)) & 
                          (df['GAME_DATE_HOME'] < match_date)]
    
    return team_matches.sort_values('GAME_DATE_HOME', ascending=False)

def calculate_win(result, is_home_team):
    return 1 if (result == 'HOME' and is_home_team) or (result == 'AWAY' and not is_home_team) else 0

def get_team_form(df, team, match_date, n_matches, venue):
    team_matches = get_recent_team_stats(df, team, match_date, venue)
    if len(team_matches) == 0:
        return [0]*19
    
    recent_matches = team_matches.head(n_matches)
    wins = losses = pts_for = pts_against = reb_for = reb_against = ast_for = ast_against = 0
    stl_for = stl_against = blk_for = blk_against = tov_for = tov_against = pm_for = pm_against = 0
    fg_pct = fg3_pct = ft_pct = 0
    
    for _, match in recent_matches.iterrows():
        is_home = match['TEAM_NAME_HOME'] == team
        win = calculate_win(match['WINNER'], is_home)
        wins += win
        losses += (1 - win)
        
        if is_home:
            pts_for += match['PTS_HOME']; pts_against += match['PTS_AWAY']
            reb_for += match['REB_HOME']; reb_against += match['REB_AWAY']
            ast_for += match['AST_HOME']; ast_against += match['AST_AWAY']
            stl_for += match['STL_HOME']; stl_against += match['STL_AWAY']
            blk_for += match['BLK_HOME']; blk_against += match['BLK_AWAY']
            tov_for += match['TOV_HOME']; tov_against += match['TOV_AWAY']
            pm_for += match['PLUS_MINUS_HOME']; pm_against += match['PLUS_MINUS_AWAY']
            fg_pct += match['FG_PCT_HOME']; fg3_pct += match['FG3_PCT_HOME']; ft_pct += match['FT_PCT_HOME']
        else:
            pts_for += match['PTS_AWAY']; pts_against += match['PTS_HOME']
            reb_for += match['REB_AWAY']; reb_against += match['REB_HOME']
            ast_for += match['AST_AWAY']; ast_against += match['AST_HOME']
            stl_for += match['STL_AWAY']; stl_against += match['STL_HOME']
            blk_for += match['BLK_AWAY']; blk_against += match['BLK_HOME']
            tov_for += match['TOV_AWAY']; tov_against += match['TOV_HOME']
            pm_for += match['PLUS_MINUS_AWAY']; pm_against += match['PLUS_MINUS_HOME']
            fg_pct += match['FG_PCT_AWAY']; fg3_pct += match['FG3_PCT_AWAY']; ft_pct += match['FT_PCT_AWAY']
    
    return (wins, losses, pts_for, pts_against, reb_for, reb_against, ast_for, ast_against,
            stl_for, stl_against, blk_for, blk_against, tov_for, tov_against, pm_for, pm_against,
            fg_pct, fg3_pct, ft_pct)

def get_elo_ratings(elo_home, elo_away, result, k=20):
    exp_home = 1 / (1 + 10 ** ((elo_away - elo_home) / 400))
    exp_away = 1 - exp_home
    score_home, score_away = (1,0) if result == 'HOME' else (0,1)
    elo_home = elo_home + k * (score_home - exp_home)
    elo_away = elo_away + k * (score_away - exp_away)
    return elo_home, elo_away

def update_attack_and_defense_elo(attack_elo_home, defense_elo_home,
                                  attack_elo_away, defense_elo_away,
                                  home_pts, away_pts, k=10):
    exp_home_pts = (attack_elo_home - defense_elo_away) / 400 + 100
    exp_away_pts = (attack_elo_away - defense_elo_home) / 400 + 100
    
    new_attack_elo_home = attack_elo_home + k * (home_pts - exp_home_pts)
    new_attack_elo_away = attack_elo_away + k * (away_pts - exp_away_pts)
    new_defense_elo_home = defense_elo_home + k * (exp_away_pts - away_pts)
    new_defense_elo_away = defense_elo_away + k * (exp_home_pts - home_pts)
    
    return new_attack_elo_home, new_defense_elo_home, new_attack_elo_away, new_defense_elo_away


In [4]:
# ---------------------
# Feature Engineering
# ---------------------

def create_match_features(df, match_idx, elo, attack_elo, defense_elo):
    match = df.iloc[match_idx]
    home_team, away_team = match['TEAM_NAME_HOME'], match['TEAM_NAME_AWAY']
    match_date = match['GAME_DATE_HOME']

    current_elo_home, current_elo_away = elo[home_team], elo[away_team]
    elo_diff = current_elo_home - current_elo_away
    current_attack_elo_home, current_defense_elo_home = attack_elo[home_team], defense_elo[home_team]
    current_attack_elo_away, current_defense_elo_away = attack_elo[away_team], defense_elo[away_team]

    # Forma zadnjih 5 utakmica
    (home_wins, home_losses, home_pts_for, home_pts_against, home_reb_for, home_reb_against,
     home_ast_for, home_ast_against, home_stl_for, home_stl_against, home_blk_for, home_blk_against,
     home_tov_for, home_tov_against, home_pm_for, home_pm_against, home_fg_pct, home_fg3_pct, home_ft_pct) = get_team_form(df, home_team, match_date, 5, 'any')
    
    (away_wins, away_losses, away_pts_for, away_pts_against, away_reb_for, away_reb_against,
     away_ast_for, away_ast_against, away_stl_for, away_stl_against, away_blk_for, away_blk_against,
     away_tov_for, away_tov_against, away_pm_for, away_pm_against, away_fg_pct, away_fg3_pct, away_ft_pct) = get_team_form(df, away_team, match_date, 5, 'any')

    features = {
        # Home team averages
        'home_win_rate': home_wins / 5,
        'home_loss_rate': home_losses / 5,
        'home_avg_pts_scored': home_pts_for / 5,
        'home_avg_pts_conceded': home_pts_against / 5,
        'home_avg_reb_for': home_reb_for / 5,
        'home_avg_reb_against': home_reb_against / 5,
        'home_avg_ast_for': home_ast_for / 5,
        'home_avg_ast_against': home_ast_against / 5,
        'home_avg_stl_for': home_stl_for / 5,
        'home_avg_stl_against': home_stl_against / 5,
        'home_avg_blk_for': home_blk_for / 5,
        'home_avg_blk_against': home_blk_against / 5,
        'home_avg_tov_for': home_tov_for / 5,
        'home_avg_tov_against': home_tov_against / 5,
        'home_avg_pm_for': home_pm_for / 5,
        'home_avg_pm_against': home_pm_against / 5,
        'home_avg_fg_pct': home_fg_pct / 5,
        'home_avg_fg3_pct': home_fg3_pct / 5,
        'home_avg_ft_pct': home_ft_pct / 5,

        # Away team averages
        'away_win_rate': away_wins / 5,
        'away_loss_rate': away_losses / 5,
        'away_avg_pts_scored': away_pts_for / 5,
        'away_avg_pts_conceded': away_pts_against / 5,
        'away_avg_reb_for': away_reb_for / 5,
        'away_avg_reb_against': away_reb_against / 5,
        'away_avg_ast_for': away_ast_for / 5,
        'away_avg_ast_against': away_ast_against / 5,
        'away_avg_stl_for': away_stl_for / 5,
        'away_avg_stl_against': away_stl_against / 5,
        'away_avg_blk_for': away_blk_for / 5,
        'away_avg_blk_against': away_blk_against / 5,
        'away_avg_tov_for': away_tov_for / 5,
        'away_avg_tov_against': away_tov_against / 5,
        'away_avg_pm_for': away_pm_for / 5,
        'away_avg_pm_against': away_pm_against / 5,
        'away_avg_fg_pct': away_fg_pct / 5,
        'away_avg_fg3_pct': away_fg3_pct / 5,
        'away_avg_ft_pct': away_ft_pct / 5,

        # Elo ratings
        'elo_home': current_elo_home,
        'elo_away': current_elo_away,
        'elo_diff': elo_diff,
        'attack_elo_home': current_attack_elo_home,
        'defense_elo_home': current_defense_elo_home,
        'attack_elo_away': current_attack_elo_away,
        'defense_elo_away': current_defense_elo_away
    }
    
    new_elo_home, new_elo_away = get_elo_ratings(current_elo_home, current_elo_away, match['WINNER'])
    elo[home_team] = new_elo_home
    elo[away_team] = new_elo_away
    
    new_attack_elo_home, new_defense_elo_home, new_attack_elo_away, new_defense_elo_away = update_attack_and_defense_elo(
        current_attack_elo_home, current_defense_elo_home,
        current_attack_elo_away, current_defense_elo_away,
        match['PTS_HOME'], match['PTS_AWAY']
    )
    attack_elo[home_team] = new_attack_elo_home
    defense_elo[home_team] = new_defense_elo_home
    attack_elo[away_team] = new_attack_elo_away
    defense_elo[away_team] = new_defense_elo_away
    
    return features


In [5]:
# --------------------------------------------
# Applying feature engineering to all matches
# --------------------------------------------

print("Creating features for all NBA matches...")
print("This may take several minutes...")

teams = set(df['TEAM_NAME_HOME']).union(set(df['TEAM_NAME_AWAY']))
elo = {team: 1500 for team in teams}
attack_elo = {team: 1500 for team in teams}
defense_elo = {team: 1500 for team in teams}

features_list = []

for idx in tqdm.tqdm(range(len(df))):
    try:
        features = create_match_features(df, idx, elo, attack_elo, defense_elo)
        features_list.append(features)
    except Exception as e:
        print(f"Error at match {idx}: {e}")
        if features_list:
            empty_features = {key: 0 for key in features_list[0].keys()}
            features_list.append(empty_features)

df_features = pd.DataFrame(features_list)

print(f"\nFeatures created: {df_features.shape[0]} rows, {df_features.shape[1]} columns")
print(f"\nFeature columns:")
print(df_features.columns.tolist())
print(f"\nFirst row:")
print(df_features.iloc[0])
print(f"\nCheck for NaN values:")
print(df_features.isnull().sum().sum())


Creating features for all NBA matches...
This may take several minutes...


  0%|          | 0/8048 [00:00<?, ?it/s]

100%|██████████| 8048/8048 [00:55<00:00, 144.69it/s]


Features created: 8048 rows, 45 columns

Feature columns:
['home_win_rate', 'home_loss_rate', 'home_avg_pts_scored', 'home_avg_pts_conceded', 'home_avg_reb_for', 'home_avg_reb_against', 'home_avg_ast_for', 'home_avg_ast_against', 'home_avg_stl_for', 'home_avg_stl_against', 'home_avg_blk_for', 'home_avg_blk_against', 'home_avg_tov_for', 'home_avg_tov_against', 'home_avg_pm_for', 'home_avg_pm_against', 'home_avg_fg_pct', 'home_avg_fg3_pct', 'home_avg_ft_pct', 'away_win_rate', 'away_loss_rate', 'away_avg_pts_scored', 'away_avg_pts_conceded', 'away_avg_reb_for', 'away_avg_reb_against', 'away_avg_ast_for', 'away_avg_ast_against', 'away_avg_stl_for', 'away_avg_stl_against', 'away_avg_blk_for', 'away_avg_blk_against', 'away_avg_tov_for', 'away_avg_tov_against', 'away_avg_pm_for', 'away_avg_pm_against', 'away_avg_fg_pct', 'away_avg_fg3_pct', 'away_avg_ft_pct', 'elo_home', 'elo_away', 'elo_diff', 'attack_elo_home', 'defense_elo_home', 'attack_elo_away', 'defense_elo_away']

First row:
home_win

In [ ]:
# --------------------------------------------------------------------
# Combining features with original data and creating final NBA dataset
# --------------------------------------------------------------------

df_features.reset_index(drop=True, inplace=True)
df.reset_index(drop=True, inplace=True)

df_final = pd.concat([
    df[['GAME_DATE_HOME', 'TEAM_NAME_HOME', 'TEAM_NAME_AWAY', 'WINNER', 'season']], 
    df_features
], axis=1)

print(f"\nFinal NBA dataset:")
print(f"  Shape: {df_final.shape}")
print(f"  Seasons: {sorted(df_final['season'].unique())}")
print(f"  Missing values: {df_final.isnull().sum().sum()}")

df_final.to_csv('../data/processed/nba_dataset_final_a.csv', index=False)



Final NBA dataset:
  Shape: (15108, 50)
  Seasons: ['201415', '201516', '201617', '201718', '201819', '201920', '202021', '202122', '202223', '202324', '202425']
  Missing values: 0


In [6]:
# --------------------------------------------------------------------
# Combining features with original data and creating final NBA dataset
# --------------------------------------------------------------------

df_features.reset_index(drop=True, inplace=True)
df.reset_index(drop=True, inplace=True)

df_final = pd.concat([
    df[['GAME_DATE_HOME', 'TEAM_NAME_HOME', 'TEAM_NAME_AWAY', 'WINNER', 'season']], 
    df_features
], axis=1)

print(f"\nFinal NBA dataset:")
print(f"  Shape: {df_final.shape}")
print(f"  Seasons: {sorted(df_final['season'].unique())}")
print(f"  Missing values: {df_final.isnull().sum().sum()}")

df_final.to_csv('../data/processed/nba_dataset_final_b.csv', index=False)



Final NBA dataset:
  Shape: (8048, 50)
  Seasons: ['201920', '202021', '202122', '202223', '202324', '202425']
  Missing values: 0
